# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Veduisback/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — Growth prediction

The FlyRank research paper reports high accuracy for growth prediction, including separate results for unseen pages from the same brands and for unseen brands.

My methodology question is: **where exactly does the growth label come from, and does the validation design support the scope of that accuracy claim?**

This matters because a model can perform differently when tested on pages from known brands versus pages from entirely unseen brands. The paper does report these evaluation settings separately, which is useful. I would still want to verify that the label is defined from information available only after the prediction point and that the validation population matches the population implied by the claim.

I am treating this as a methodology question rather than a criticism of the finding.

### Finding 2 — Refreshed vs stale content

The paper reports a statistically significant difference in impressions between refreshed and stale content using a Mann–Whitney U test.

My methodology question is: **does this comparison measure an observed association, or could differences between refreshed and stale pages contribute to the result?**

For example, content age and other page characteristics may differ between the two groups. A significant difference in the measured distributions does not by itself establish that refreshing the content caused the difference.

The paper itself describes its analysis as identifying patterns rather than proving causation. My audit follows the same standard: I would describe the result as an observed association unless the study design provides stronger evidence for a causal interpretation.

In [8]:
# Section 1: simple audit record for the two research findings

import pandas as pd

paper_findings = [
    {
        "finding": "Growth prediction accuracy",
        "methodology_question": (
            "Where does the growth label come from, and does the "
            "validation design support the scope of the accuracy claim?"
        ),
    },
    {
        "finding": "Refreshed vs stale impressions",
        "methodology_question": (
            "Does the comparison establish an observed association, "
            "or could differences between the groups contribute to it?"
        ),
    },
]

paper_audit = pd.DataFrame(paper_findings)

print("Paper findings audited:", len(paper_audit))
display(paper_audit)

Paper findings audited: 2


,finding,methodology_question
0,Growth prediction accuracy,"Where does the growth label come from, and doe..."
1,Refreshed vs stale impressions,Does the comparison establish an observed asso...


In [10]:
# Section 2: Load the exact W05 modeling population

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF token loaded:", HF_TOKEN is not None)

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("INSTALL azure")
con.execute("LOAD azure")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

DECISION_DATE = "2026-02-28"

FEB_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-02/*.parquet"
)

MARCH_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

DIM_CONTENT_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "dim_content.parquet"
)

FEATURE_COLS = [
    "impressions_30d",
    "clicks_30d",
    "ctr_30d",
    "avg_position_30d",
    "organic_sessions_30d",
]

# ------------------------------------------------------------
# February decision-time features
# ------------------------------------------------------------

feature_sql = f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        sessions_organic
    FROM read_parquet('{FEB_PATH}')
    WHERE gsc_data_available IS TRUE
),

performance AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_30d,

        SUM(gsc_clicks) AS clicks_30d,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr_30d,

        AVG(gsc_avg_position) AS avg_position_30d,

        SUM(sessions_organic) AS organic_sessions_30d

    FROM daily

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM performance
"""

features_df = con.execute(feature_sql).df()


# ------------------------------------------------------------
# March future-outcome proxy
# ------------------------------------------------------------

outcome_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,

    CASE
        WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
        THEN 1
        ELSE 0
    END AS march_had_impressions

FROM read_parquet('{MARCH_PATH}')

GROUP BY
    client_hash_id,
    content_hash_id
"""

outcome_df = con.execute(outcome_sql).df()


# ------------------------------------------------------------
# Exact W04 eligibility population
# ------------------------------------------------------------

eligibility_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    content_updated_date,
    COALESCE(search_volume, 0) AS search_volume

FROM read_parquet('{DIM_CONTENT_PATH}')

WHERE content_updated_date IS NOT NULL
  AND content_updated_date <= DATE '{DECISION_DATE}'
"""

eligibility_df = con.execute(eligibility_sql).df()


# ------------------------------------------------------------
# Combine everything
# ------------------------------------------------------------

model_df = (
    features_df
    .merge(
        outcome_df,
        on=["client_hash_id", "content_hash_id"],
        how="left"
    )
    .merge(
        eligibility_df,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
)

model_df["march_had_impressions"] = (
    model_df["march_had_impressions"]
    .fillna(0)
    .astype(int)
)

model_df = model_df.dropna(
    subset=FEATURE_COLS + ["march_had_impressions"]
).copy()


print("W04-eligible model frame shape:", model_df.shape)
print("Unique clients:", model_df["client_hash_id"].nunique())

print("\nTarget distribution:")

display(
    model_df["march_had_impressions"]
    .value_counts()
    .sort_index()
    .rename_axis("march_had_impressions")
    .reset_index(name="n")
)

HF token loaded: True


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

W04-eligible model frame shape: (16309, 10)
Unique clients: 17

Target distribution:


,march_had_impressions,n
0,0,1870
1,1,14439


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

In Week 5, the final model was evaluated using a grouped split by client, so pages from the same client did not appear in both train and test sets. This better matches the question of whether the model can generalize to a client that was not represented during training.

For the audit, I compare that result with a row-random split using the same eligible population, features, model, and target. The random split allows rows from the same client to appear in both train and test sets, so it measures a different and potentially easier generalization setting.

The **before** result is the row-random split. The **after** result is the grouped-by-client split.

I treat the grouped result as the more relevant validation design for an unseen-client generalization question. The comparison does not prove production performance; it shows how the measured estimate changes when the split better reflects the intended validation setting.


### Before/after result

Under a random row split, the model measured ROC-AUC of **0.8989** and Average Precision of **0.9860**. However, 15 clients appeared in both the training and test sets, so this split does not test generalization to unseen clients.

Under the grouped-by-client split, ROC-AUC was **0.7179** and Average Precision was **0.9642**. There were 13 training clients and 4 test clients, with zero client overlap.

The difference shows why the validation design matters. The random split produced a higher measured estimate, but it answered a different question because the same clients were represented on both sides of the split. The grouped split gives a more direct measurement for the intended unseen-client validation setting.

I therefore use the grouped result as the validation result for this audit rather than presenting the random-split performance as evidence of unseen-client generalization.


In [11]:
# Section 2: Compare random row split vs grouped-by-client split

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

X = model_df[FEATURE_COLS]
y = model_df["march_had_impressions"]
groups = model_df["client_hash_id"]

# ------------------------------------------------------------
# BEFORE: random row split
# ------------------------------------------------------------
X_train_random, X_test_random, y_train_random, y_test_random, idx_train_random, idx_test_random = (
    train_test_split(
        X,
        y,
        model_df.index,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

random_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

random_model.fit(X_train_random, y_train_random)

random_prob = random_model.predict_proba(X_test_random)[:, 1]

random_auc = roc_auc_score(y_test_random, random_prob)
random_ap = average_precision_score(y_test_random, random_prob)

random_train_clients = set(
    model_df.loc[idx_train_random, "client_hash_id"]
)
random_test_clients = set(
    model_df.loc[idx_test_random, "client_hash_id"]
)

random_client_overlap = random_train_clients & random_test_clients


# ------------------------------------------------------------
# AFTER: grouped-by-client split
# ------------------------------------------------------------
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df[FEATURE_COLS],
        model_df["march_had_impressions"],
        groups=model_df["client_hash_id"]
    )
)

group_train = model_df.iloc[train_idx].copy()
group_test = model_df.iloc[test_idx].copy()

group_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

group_model.fit(
    group_train[FEATURE_COLS],
    group_train["march_had_impressions"]
)

group_prob = group_model.predict_proba(
    group_test[FEATURE_COLS]
)[:, 1]

group_auc = roc_auc_score(
    group_test["march_had_impressions"],
    group_prob
)

group_ap = average_precision_score(
    group_test["march_had_impressions"],
    group_prob
)

group_train_clients = set(group_train["client_hash_id"])
group_test_clients = set(group_test["client_hash_id"])

group_client_overlap = group_train_clients & group_test_clients


# ------------------------------------------------------------
# Comparison
# ------------------------------------------------------------
comparison = pd.DataFrame([
    {
        "split": "Before: random rows",
        "train_rows": len(X_train_random),
        "test_rows": len(X_test_random),
        "train_clients": len(random_train_clients),
        "test_clients": len(random_test_clients),
        "client_overlap": len(random_client_overlap),
        "roc_auc": random_auc,
        "average_precision": random_ap,
    },
    {
        "split": "After: grouped by client",
        "train_rows": len(group_train),
        "test_rows": len(group_test),
        "train_clients": len(group_train_clients),
        "test_clients": len(group_test_clients),
        "client_overlap": len(group_client_overlap),
        "roc_auc": group_auc,
        "average_precision": group_ap,
    }
])

print("Model validation comparison")
print("=" * 70)

display(
    comparison.style.format({
        "roc_auc": "{:.4f}",
        "average_precision": "{:.4f}"
    })
)

print("\nClient overlap check:")
print("Random split overlap:", len(random_client_overlap))
print("Grouped split overlap:", len(group_client_overlap))

print("\nGrouped test clients:")
print(sorted(group_test_clients))

Model validation comparison


,split,train_rows,test_rows,train_clients,test_clients,client_overlap,roc_auc,average_precision
0,Before: random rows,13047,3262,16,16,15,0.8989,0.9860
1,After: grouped by client,13693,2616,13,4,0,0.7179,0.9642



Client overlap check:
Random split overlap: 15
Grouped split overlap: 0

Grouped test clients:
['client_08d2847f24cf89c1', 'client_0e1acc6cd57b0eba', 'client_400c21c81c8b46ef', 'client_f623b01661d4bfe4']


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

The final model uses five decision-time features: February impressions, clicks, CTR, average position, and organic sessions.

These features are intended to represent information available by the February 28 decision cutoff. The outcome proxy, `march_had_impressions`, is calculated from March data and is therefore kept separate from the feature set.

I also check the final feature names for explicit future/outcome terms and verify that the target is not included among the model features.

This is a feature-level audit rather than proof that every upstream data transformation is leakage-free. The main timing boundary is February features versus the March outcome.


In [12]:
# Section 3: Final feature leakage audit

print("FINAL FEATURE LEAKAGE AUDIT")
print("=" * 70)

# The target must never be part of the feature set.
target_col = "march_had_impressions"

print("\n1. Target separation")
print("Target:", target_col)
print("Feature columns:", FEATURE_COLS)
print("Target included in features:", target_col in FEATURE_COLS)

assert target_col not in FEATURE_COLS


# ------------------------------------------------------------
# Check feature names for obvious future/outcome indicators
# ------------------------------------------------------------

future_tokens = [
    "march",
    "future",
    "target",
    "outcome",
    "had_impressions"
]

feature_name_audit = []

for feature in FEATURE_COLS:
    lower_name = feature.lower()

    matched_tokens = [
        token
        for token in future_tokens
        if token in lower_name
    ]

    feature_name_audit.append({
        "feature": feature,
        "future/outcome_token_found": len(matched_tokens) > 0,
        "matched_tokens": ", ".join(matched_tokens) if matched_tokens else ""
    })

feature_name_audit_df = pd.DataFrame(feature_name_audit)

print("\n2. Feature-name audit")
display(feature_name_audit_df)


# ------------------------------------------------------------
# Confirm the actual model frame contains the expected columns
# ------------------------------------------------------------

expected_columns = FEATURE_COLS + [
    "march_had_impressions"
]

missing_columns = [
    col for col in expected_columns
    if col not in model_df.columns
]

print("\n3. Required-column check")
print("Missing expected columns:", missing_columns)

assert len(missing_columns) == 0


# ------------------------------------------------------------
# Confirm the five features are numeric
# ------------------------------------------------------------

print("\n4. Feature data types")

dtype_audit = pd.DataFrame({
    "feature": FEATURE_COLS,
    "dtype": [
        str(model_df[col].dtype)
        for col in FEATURE_COLS
    ],
    "missing_values": [
        int(model_df[col].isna().sum())
        for col in FEATURE_COLS
    ]
})

display(dtype_audit)


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

name_leaks = feature_name_audit_df[
    feature_name_audit_df["future/outcome_token_found"]
]

print("\n5. Final audit result")
print("Number of final features:", len(FEATURE_COLS))
print("Features with obvious future/outcome names:", len(name_leaks))
print("Target included in feature set:", target_col in FEATURE_COLS)
print("Decision-time feature window: February 2026")
print("Future outcome window: March 2026")

if len(name_leaks) == 0 and target_col not in FEATURE_COLS:
    print("\nRESULT: No obvious feature-name leakage detected.")
else:
    print("\nRESULT: Review the flagged features before submission.")

FINAL FEATURE LEAKAGE AUDIT

1. Target separation
Target: march_had_impressions
Feature columns: ['impressions_30d', 'clicks_30d', 'ctr_30d', 'avg_position_30d', 'organic_sessions_30d']
Target included in features: False

2. Feature-name audit


,feature,future/outcome_token_found,matched_tokens
0,impressions_30d,False,
1,clicks_30d,False,
2,ctr_30d,False,
3,avg_position_30d,False,
4,organic_sessions_30d,False,



3. Required-column check
Missing expected columns: []

4. Feature data types


,feature,dtype,missing_values
0,impressions_30d,float64,0
1,clicks_30d,float64,0
2,ctr_30d,float64,0
3,avg_position_30d,float64,0
4,organic_sessions_30d,float64,0



5. Final audit result
Number of final features: 5
Features with obvious future/outcome names: 0
Target included in feature set: False
Decision-time feature window: February 2026
Future outcome window: March 2026

RESULT: No obvious feature-name leakage detected.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original bold claim

On this held-out client split, Logistic Regression produced stronger measured ranking performance than the W04 baseline against the `march_had_impressions` outcome proxy.

### Safer claim

On the held-out client split, the Logistic Regression model **measured** ROC-AUC of 0.7179 and Average Precision of 0.9642 against the `march_had_impressions` outcome proxy. This provides an **observed, directional ranking signal** that could support **decision-support** for prioritizing pages for human review.

The result does not establish that a page needs a refresh, that refreshing will improve performance, or that the model will produce the same results in production. The outcome is a proxy based on March impressions, so the measured result should be interpreted within that scope.


In [13]:
# Section 4: Record the measured result used in the claim rewrite

claim_audit = pd.DataFrame([
    {
        "metric": "ROC-AUC",
        "value": group_auc,
        "validation": "Grouped by client",
        "outcome": "march_had_impressions"
    },
    {
        "metric": "Average Precision",
        "value": group_ap,
        "validation": "Grouped by client",
        "outcome": "march_had_impressions"
    }
])

print("Measured result supporting the rewritten claim:")
display(
    claim_audit.style.format({
        "value": "{:.4f}"
    })
)

print("\nClaim scope:")
print("- Observed/measured: yes")
print("- Directional ranking signal: yes")
print("- Decision-support framing: yes")
print("- Causal refresh claim: no")
print("- Production-performance claim: no")

Measured result supporting the rewritten claim:


,metric,value,validation,outcome
0,ROC-AUC,0.7179,Grouped by client,march_had_impressions
1,Average Precision,0.9642,Grouped by client,march_had_impressions



Claim scope:
- Observed/measured: yes
- Directional ranking signal: yes
- Decision-support framing: yes
- Causal refresh claim: no
- Production-performance claim: no


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.